In [ ]:
import pickle
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, precision_recall_fscore_support, confusion_matrix
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torch.optim import AdamW
from torch.amp import autocast, GradScaler
from transformers import BigBirdTokenizer, BigBirdModel
from transformers import get_linear_schedule_with_warmup
import numpy as np
import pandas as pd
import os, json

torch.backends.cudnn.benchmark = True

In [ ]:
with open('Data/final_train_df.pkl', 'rb') as f:
    train_df = pickle.load(f)
    train_df.rename(columns={'xgb_oof_pred': 'xgb_pred'}, inplace=True)

with open('Data/final_val_df.pkl', 'rb') as f:
    val_df = pickle.load(f)

with open('Data/final_test_df.pkl', 'rb') as f:
    test_df = pickle.load(f)

In [ ]:
# Find the median of days_in_custody from train_df
median_days = train_df['days_in_custody'].median()

# Fill NaN values in all three dataframes with the median
train_df.fillna({'days_in_custody': median_days}, inplace=True)
val_df.fillna({'days_in_custody': median_days}, inplace=True)
test_df.fillna({'days_in_custody': median_days}, inplace=True)

In [ ]:
# Find medians for age columns in train_df
median_min_age = train_df['min_age'].median()
median_max_age = train_df['max_age'].median()
median_median_age = train_df['median_age'].median()

# Fill NaN values in all three dataframes
for df in [train_df, val_df, test_df]:
    df.fillna({'min_age': median_min_age}, inplace=True)
    df.fillna({'max_age': median_max_age}, inplace=True)
    df.fillna({'median_age': median_median_age}, inplace=True)

In [ ]:
train_df.columns

In [ ]:
scaler = StandardScaler()

columns_to_standardize = ['days_in_custody', 'min_age', 'max_age', 'median_age',
                          'shap_sum_pos', 'shap_sum_neg', 'shap_max_pos', 'shap_min_neg',
                          'shap_pos_count', 'shap_neg_count', 'shap_l1_total',
                          'shap_top3_abs_sum']
                          
train_df[columns_to_standardize] = scaler.fit_transform(train_df[columns_to_standardize])
val_df[columns_to_standardize] = scaler.transform(val_df[columns_to_standardize])
test_df[columns_to_standardize] = scaler.transform(test_df[columns_to_standardize])

In [ ]:
SCALAR_COLS = [
    "bail_type",
    "days_in_custody_available",
    "days_in_custody",
    "age_available",
    "min_age",
    "max_age",
    "median_age",
    # "shap_sum_pos",
    # "shap_sum_neg",
    # "shap_max_pos",
    # "shap_min_neg",
    # "shap_pos_count",
    # "shap_neg_count",
    # "shap_l1_total",
    # "shap_top3_abs_sum",
    # "xgb_pred",
]

In [ ]:
class ScalarDataset(Dataset):
    def __init__(self, df):
        self.X = df[SCALAR_COLS].astype(np.float32).to_numpy()
        self.y = df['outcome'].astype(np.float32).to_numpy()

    def __len__(self):
        return len(self.y)
    
    def __getitem__(self, idx):
        return {
            'x': torch.from_numpy(self.X[idx]),
            'y': torch.tensor(self.y[idx])
        }

In [ ]:
class TinyMLP(nn.Module):
    def __init__(self, input_dim):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(input_dim, 8),
            nn.ReLU(),
            nn.Dropout(0.5),
            nn.Linear(8, 1)
        )

    def forward(self, x):
        return self.net(x).squeeze(1)


In [ ]:
train_scalar_ds = ScalarDataset(train_df)
val_scalar_ds = ScalarDataset(val_df)

train_scalar_loader = DataLoader(train_scalar_ds, batch_size=128, shuffle=True)
val_scalar_loader = DataLoader(val_scalar_ds, batch_size=128, shuffle=False)

In [ ]:
def train_one_epoch_scalar(model, loader, optimizer, criterion, device):
    model.train()
    total_loss = 0

    for batch in loader:
        x = batch["x"].to(device)
        y = batch["y"].to(device)

        optimizer.zero_grad()
        logits = model(x)
        loss = criterion(logits, y)
        loss.backward()
        optimizer.step()

        total_loss += loss.item() * y.size(0)

    return total_loss / len(loader.dataset)


@torch.no_grad()
def eval_scalar(model, loader, criterion, device):
    model.eval()
    total_loss = 0
    all_logits = []
    all_labels = []

    for batch in loader:
        x = batch["x"].to(device)
        y = batch["y"].to(device)

        logits = model(x)
        loss = criterion(logits, y)

        total_loss += loss.item() * y.size(0)
        all_logits.append(logits.cpu())
        all_labels.append(y.cpu())

    logits = torch.cat(all_logits).numpy()
    labels = torch.cat(all_labels).numpy()

    probs = 1 / (1 + np.exp(-logits))
    preds = (probs >= 0.5).astype(int)

    acc = accuracy_score(labels, preds)
    prec, rec, f1, _ = precision_recall_fscore_support(labels, preds, average="binary")

    return {
        "loss": total_loss / len(loader.dataset),
        "accuracy": acc,
        "precision": prec,
        "recall": rec,
        "f1": f1
    }


In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, f1_score

X_train = train_df[SCALAR_COLS].values
y_train = train_df["outcome"].values

X_val = val_df[SCALAR_COLS].values
y_val = val_df["outcome"].values

clf = LogisticRegression(max_iter=2000, class_weight="balanced")
clf.fit(X_train, y_train)

val_probs = clf.predict_proba(X_val)[:,1]
val_preds = (val_probs >= 0.5).astype(int)

print("LogReg acc:", accuracy_score(y_val, val_preds))
print("LogReg f1 :", f1_score(y_val, val_preds))


In [ ]:
class LinearBaseline(nn.Module):
    def __init__(self, d):
        super().__init__()
        self.fc = nn.Linear(d, 1)

    def forward(self, x):
        return self.fc(x).squeeze(1)


In [ ]:
# =========================
# BASELINE 2 — Linear Torch
# =========================

print("\n==============================")
print("Baseline 2: Linear Torch Model")
print("==============================")

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

model_linear = LinearBaseline(len(SCALAR_COLS)).to(device)

criterion = nn.BCEWithLogitsLoss()
optimizer = torch.optim.AdamW(
    model_linear.parameters(),
    lr=1e-3,
    weight_decay=1e-2   # 🔴 strong regularization
)

best_f1 = 0
best_state = None

for epoch in range(50):
    train_loss = train_one_epoch_scalar(model_linear, train_scalar_loader, optimizer, criterion, device)
    metrics = eval_scalar(model_linear, val_scalar_loader, criterion, device)

    print(f"[Linear] Epoch {epoch:02d} | "
          f"train_loss={train_loss:.4f} | "
          f"val_loss={metrics['loss']:.4f} | "
          f"acc={metrics['accuracy']:.4f} | "
          f"f1={metrics['f1']:.4f}")

    if metrics["f1"] > best_f1:
        best_f1 = metrics["f1"]
        best_state = model_linear.state_dict()

# Load best
model_linear.load_state_dict(best_state)

print("\n✅ Best Linear Baseline F1:", best_f1)


In [ ]:
# =========================
# BASELINE 3 — Tiny MLP
# =========================

print("\n==============================")
print("Baseline 3: Tiny MLP")
print("==============================")

model_mlp = TinyMLP(len(SCALAR_COLS)).to(device)

criterion = nn.BCEWithLogitsLoss()
optimizer = torch.optim.AdamW(
    model_mlp.parameters(),
    lr=3e-4,
    weight_decay=5e-2   # 🔴 VERY strong regularization
)

best_f1 = 0
best_state = None
bad_epochs = 0
patience = 8

for epoch in range(50):
    train_loss = train_one_epoch_scalar(model_mlp, train_scalar_loader, optimizer, criterion, device)
    metrics = eval_scalar(model_mlp, val_scalar_loader, criterion, device)

    print(f"[TinyMLP] Epoch {epoch:02d} | "
          f"train_loss={train_loss:.4f} | "
          f"val_loss={metrics['loss']:.4f} | "
          f"acc={metrics['accuracy']:.4f} | "
          f"f1={metrics['f1']:.4f}")

    if metrics["f1"] > best_f1:
        best_f1 = metrics["f1"]
        best_state = model_mlp.state_dict()
        bad_epochs = 0
    else:
        bad_epochs += 1

    if bad_epochs >= patience:
        print("🛑 Early stopping TinyMLP")
        break

# Load best
model_mlp.load_state_dict(best_state)

print("\n✅ Best TinyMLP Baseline F1:", best_f1)
